In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

print("Ready")


Ready


In [3]:
def estimate_spread(ticker, start="2023-01-01", end="2024-01-01"):
    """Estimate bid-ask spread using high-low proxy"""
    df=yf.download(ticker, start=start, end=end, auto_adjust=True)
    df.columns=df.columns.get_level_values(0)

    hl_ratio =np.log(df["High"] / df["Low"])
    spread_est =2 * (np.exp(hl_ratio /2) -1) / (1 + np.exp(hl_ratio / 2))

    avg_spread = spread_est.mean()
    median_spread = spread_est.median()
    avg_price =df["Close"].mean()

    print(f"\n{ticker} SPREAD ESTIMATE")
    print(f" Avg spread (%) : {avg_spread:.4%}")
    print(f" Median spread (%) : {median_spread:.4%}")
    print(f" Avg price : ${avg_price:.2f}")
    print(f" Spread in dollars : ${avg_price * avg_spread:.4f}")
    return avg_spread

for t in ["AAPL", "SPY", "GME", "BTC-USD"]:
    estimate_spread(t)
    
    

[*********************100%***********************]  1 of 1 completed



AAPL SPREAD ESTIMATE
 Avg spread (%) : 0.8523%
 Median spread (%) : 0.7657%
 Avg price : $170.19
 Spread in dollars : $1.4505


[*********************100%***********************]  1 of 1 completed



SPY SPREAD ESTIMATE
 Avg spread (%) : 0.5243%
 Median spread (%) : 0.4754%
 Avg price : $412.35
 Spread in dollars : $2.1619


[*********************100%***********************]  1 of 1 completed



GME SPREAD ESTIMATE
 Avg spread (%) : 2.7051%
 Median spread (%) : 2.3370%
 Avg price : $19.22
 Spread in dollars : $0.5200


[*********************100%***********************]  1 of 1 completed


BTC-USD SPREAD ESTIMATE
 Avg spread (%) : 1.5099%
 Median spread (%) : 1.2424%
 Avg price : $28859.45
 Spread in dollars : $435.7503


In [9]:
def trade_cost_calculator(
    position_value,
    spread_pct   = 0.05,
    slippage_pct = 0.02,
    commission   = 0.0,
    n_trades_yr  = 50
):
    """
    Calculate round-trip transaction cost for a position.

    Parameters:
    -----------
    position_value : float — dollar value of position
    spread_pct     : float — one-way spread as % of price
    slippage_pct   : float — one-way slippage as % of price
    commission     : float — flat commission per trade ($)
    n_trades_yr    : int   — number of round trips per year
    """
    # One-way costs
    spread_cost   = position_value * (spread_pct   / 100)
    slippage_cost = position_value * (slippage_pct / 100)
    one_way_cost  = spread_cost + slippage_cost + commission

    # Round-trip
    roundtrip_cost = one_way_cost * 2
    roundtrip_pct  = roundtrip_cost / position_value * 100

    # Annual drag
    annual_cost = roundtrip_cost * n_trades_yr
    annual_drag = annual_cost / position_value * 100

    print(f"\n{'='*45}")
    print(f"  TRADE COST ANALYSIS")
    print(f"{'='*45}")
    print(f"  Position size      : ${position_value:>10,.0f}")
    print(f"  Spread cost        : ${spread_cost:>10.2f}  ({spread_pct:.3f}% one-way)")
    print(f"  Slippage cost      : ${slippage_cost:>10.2f}  ({slippage_pct:.3f}% one-way)")
    print(f"  Commission         : ${commission:>10.2f}  per side")
    print(f"  Round-trip cost    : ${roundtrip_cost:>10.2f}  ({roundtrip_pct:.3f}%)")
    print(f"  Trades per year    : {n_trades_yr:>10}")
    print(f"  Annual cost drag   : ${annual_cost:>10.2f}  ({annual_drag:.2f}% of capital)")
    print(f"{'='*45}")
    print(f"  Strategy must earn > {annual_drag:.2f}% just to break even on costs")

    return {"roundtrip_pct": roundtrip_pct, "annual_drag": annual_drag}

# Test 3 scenarios
print("\n--- SCENARIO 1: SPY swing strategy ---")
trade_cost_calculator(10000, spread_pct=0.005,
                      slippage_pct=0.01, n_trades_yr=50)

print("\n--- SCENARIO 2: Small cap momentum ---")
trade_cost_calculator(10000, spread_pct=0.5,
                      slippage_pct=0.1, n_trades_yr=100)

print("\n--- SCENARIO 3: High frequency (100 trades/day) ---")
trade_cost_calculator(10000, spread_pct=0.005,
                      slippage_pct=0.005, n_trades_yr=25000)
  


--- SCENARIO 1: SPY swing strategy ---

  TRADE COST ANALYSIS
  Position size      : $    10,000
  Spread cost        : $      0.50  (0.005% one-way)
  Slippage cost      : $      1.00  (0.010% one-way)
  Commission         : $      0.00  per side
  Round-trip cost    : $      3.00  (0.030%)
  Trades per year    :         50
  Annual cost drag   : $    150.00  (1.50% of capital)
  Strategy must earn > 1.50% just to break even on costs

--- SCENARIO 2: Small cap momentum ---

  TRADE COST ANALYSIS
  Position size      : $    10,000
  Spread cost        : $     50.00  (0.500% one-way)
  Slippage cost      : $     10.00  (0.100% one-way)
  Commission         : $      0.00  per side
  Round-trip cost    : $    120.00  (1.200%)
  Trades per year    :        100
  Annual cost drag   : $  12000.00  (120.00% of capital)
  Strategy must earn > 120.00% just to break even on costs

--- SCENARIO 3: High frequency (100 trades/day) ---

  TRADE COST ANALYSIS
  Position size      : $    10,000
  Spr

{'roundtrip_pct': 0.02, 'annual_drag': 500.0}

In [10]:
print("--- YOUR STRATEGY: Daily signal, liquid stocks ---")
trade_cost_calculator(
    position_value = 10000,
    spread_pct = 0.01,
    slippage_pct = 0.02,
    commission = 0.0,
    n_trades_yr = 100
)

--- YOUR STRATEGY: Daily signal, liquid stocks ---

  TRADE COST ANALYSIS
  Position size      : $    10,000
  Spread cost        : $      1.00  (0.010% one-way)
  Slippage cost      : $      2.00  (0.020% one-way)
  Commission         : $      0.00  per side
  Round-trip cost    : $      6.00  (0.060%)
  Trades per year    :        100
  Annual cost drag   : $    600.00  (6.00% of capital)
  Strategy must earn > 6.00% just to break even on costs


{'roundtrip_pct': 0.06, 'annual_drag': 6.0}

In [11]:
import inspect

code = """import numpy as np
import pandas as pd 
import yfinance as yf

""" + inspect.getsource(estimate_spread) + "\n\n" + inspect.getsource(trade_cost_calculator)

with open ("../utils/trade_costs.py", "w", encoding="utf-8") as f:
    f.write(code)

print("Saved to utils/trade_costs.py")

Saved to utils/trade_costs.py
